In [0]:
import json
import datetime
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

def create_order(order_id, user_id, product_id, quantity):
    try:
        # 1. Gọi Product Service để lấy giá
        print(f"Calling ProductService for product_id={product_id}...")
        price_str = dbutils.notebook.run("/Users/sv6858005@gmail.com/ProductService", 180, {"product_id": str(product_id)})
        print(f"ProductService returned: {repr(price_str)}")
        
        if price_str is None or price_str.strip() == "":
            raise ValueError("ProductService returned empty or None")
        
        price = int(price_str.strip())
        if price < 0:
            raise ValueError(f"Product {product_id} not found")
            
        amount = quantity * price
        print(f"Price={price}, Amount={amount}")
        
        # 2. Gọi Payment Service
        print(f"Calling PaymentService for order_id={order_id}, amount={amount}...")
        pay_result = dbutils.notebook.run("/Users/sv6858005@gmail.com/PaymentService", 180, {"order_id": str(order_id), "amount": str(amount)})
        print(f"PaymentService returned: {repr(pay_result)}")
        
        # 3. Gọi Notification Service
        print(f"Calling NotificationService for order_id={order_id}, user_id={user_id}...")
        notif_result = dbutils.notebook.run("/Users/sv6858005@gmail.com/NotificationService", 180, {"order_id": str(order_id), "user_id": str(user_id)})
        print(f"NotificationService returned: {repr(notif_result)}")
        
        # 4. Ghi vào bảng orders với explicit schema
        print("Writing to orders table...")
        new_order = [(order_id, user_id, product_id, quantity, amount, "completed", datetime.datetime.now().isoformat())]
        
        # Define schema to match table schema (INT for all integer fields)
        schema = StructType([
            StructField("order_id", IntegerType(), True),
            StructField("user_id", IntegerType(), True),
            StructField("product_id", IntegerType(), True),
            StructField("quantity", IntegerType(), True),
            StructField("amount", IntegerType(), True),
            StructField("status", StringType(), True),
            StructField("timestamp", StringType(), True)
        ])
        
        df_order = spark.createDataFrame(new_order, schema)
        df_order.write.mode("append").saveAsTable("soa_catalog.ecommerce.orders")
        print("Order completed successfully!")
        
        return json.dumps({"status": "SUCCESS", "order_id": order_id, "amount": amount})
    except Exception as e:
        error_msg = f"Error in create_order: {type(e).__name__}: {str(e)}"
        print(error_msg)
        import traceback
        traceback.print_exc()
        return json.dumps({"status": "ERROR", "message": error_msg})

# Nhận tham số từ widget (mô phỏng API Gateway)
dbutils.widgets.text("order_id", "1001")
dbutils.widgets.text("user_id", "1")
dbutils.widgets.text("product_id", "201")
dbutils.widgets.text("quantity", "2")

order_id = int(dbutils.widgets.get("order_id"))
user_id = int(dbutils.widgets.get("user_id"))
product_id = int(dbutils.widgets.get("product_id"))
quantity = int(dbutils.widgets.get("quantity"))

result = create_order(order_id, user_id, product_id, quantity)
print(result)

Calling ProductService for product_id=201...
ProductService returned: '150'
Price=150, Amount=300
Calling PaymentService for order_id=1001, amount=300...
PaymentService returned: 'PAID'
Calling NotificationService for order_id=1001, user_id=1...
NotificationService returned: 'NOTIFICATION sent: Order 1001 for user 1 has been processed.'
Writing to orders table...
Order completed successfully!
{"status": "SUCCESS", "order_id": 1001, "amount": 300}
